# 20. Azure Container Apps & API Management
**Duration:** 30 min | **Topics:** Deployment, auto-scaling, API gateway

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Skill | Why It Matters |
|---|-------|----------------|
| 1 | Deploy an LLM container to **Azure Container Apps** with secrets wired from Key Vault | Container Apps is the simplest path from a Dockerfile to a production HTTPS endpoint |
| 2 | Configure **HTTP concurrency auto-scaling** and understand scale-to-zero economics | Scale-to-zero eliminates idle costs — critical when running dev/staging 24/7 |
| 3 | Set up **Azure API Management** products, subscriptions, and rate-limit policies | APIM is the single front door for auth, rate limiting, cost control, and routing |
| 4 | Trace an **end-to-end authenticated API call** through APIM → ACA → LLM | Seeing the full chain helps you debug failures at the right layer |
| 5 | Perform a **canary deployment** with traffic splitting between revisions | Shipping to 10% first lets you catch regressions before 100% of users are affected |
| 6 | Execute a **30-second rollback** with `az containerapp ingress traffic set` | Fast rollback is the most important incident-response capability you can have |

---

## 📐 Architecture

```
Internet
  │  HTTPS + Ocp-Apim-Subscription-Key
  ▼
Azure API Management  (rate limit / JWT auth / route / log)
  │
  ▼
Azure Container Apps  (auto-scales 0 → N replicas on HTTP concurrency)
  │  reads AZURE_OPENAI_KEY from Key Vault at startup
  ▼
Azure OpenAI  (gpt-4o-mini)
```

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Container Apps (Consumption) | - | Free (first 180 k vCPU-s/month) |
| Azure API Management | Developer | ~\$0.07/hr — **stop when done!** |
| Azure Container Registry | Basic | ~\$0.167/day |


In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['azure-mgmt-appcontainers', 'azure-identity', 'azure-mgmt-apimanagement'])


✅ Ready: azure-mgmt-appcontainers, azure-identity, azure-mgmt-apimanagement
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Deploy Container App with Secrets and Env Vars

In [ ]:
# Container Apps — Full Deployment with Secrets
# Run commands in Azure Cloud Shell.

RG = 'rg-llm-demo'
LOCATION = 'eastus'
ACR = 'acrllmdemo'
ENV = 'aca-env-llm'
APP = 'llm-api-app'
IMAGE = 'acrllmdemo.azurecr.io/llm-api:v1.0'
KV = 'kv-llm-prod'

steps = [
    ('1. Create Container Apps environment', f'az containerapp env create --name {ENV} --resource-group {RG} --location {LOCATION}'),
    ('2. Deploy app (min-replicas=0 = free when idle)', f'az containerapp create --name {APP} --resource-group {RG} --environment {ENV} --image {IMAGE} --registry-server {ACR}.azurecr.io --target-port 8000 --ingress external --min-replicas 0 --max-replicas 10 --cpu 0.5 --memory 1.0Gi'),
    ('3. Set Key Vault secret reference (never plain-text in config)', f'az containerapp secret set --name {APP} --resource-group {RG} --secrets oai-key=keyvaultref:{KV}/azure-openai-key'),
    ('4. Wire secret to environment variable', f'az containerapp update --name {APP} --resource-group {RG} --set-env-vars AZURE_OPENAI_KEY=secretref:oai-key MODEL_NAME=gpt-4o-mini'),
    ('5. Get public URL', f'az containerapp show --name {APP} --resource-group {RG} --query properties.configuration.ingress.fqdn -o tsv'),
]
print("="*70)
print("  Container Apps — Full Deployment with Secrets")
print("="*70)
for t,c in steps:
    print(f"\n--- {t} ---")
    print(c)
print('Environment aca-env-llm provisioned')
print('App llm-api-app deployed  → 0 replicas (idle, $0 cost)')
print('Secret oai-key wired from Key Vault kv-llm-prod')
print('Env var AZURE_OPENAI_KEY → secretref:oai-key (key never in config)')
print('Public URL: https://llm-api-app.proudbeach-12345.eastus.azurecontainerapps.io')


  Container Apps — Full Deployment with Secrets

--- 1. Create Container Apps environment ---
az containerapp env create --name aca-env-llm --resource-group rg-llm-demo --location eastus

--- 2. Deploy app (min-replicas=0 = free when idle) ---
az containerapp create --name llm-api-app --resource-group rg-llm-demo --environment aca-env-llm --image acrllmdemo.azurecr.io/llm-api:v1.0 --registry-server acrllmdemo.azurecr.io --target-port 8000 --ingress external --min-replicas 0 --max-replicas 10 --cpu 0.5 --memory 1.0Gi

--- 3. Set Key Vault secret reference (never plain-text in config) ---
az containerapp secret set --name llm-api-app --resource-group rg-llm-demo --secrets oai-key=keyvaultref:kv-llm-prod/azure-openai-key

--- 4. Wire secret to environment variable ---
az containerapp update --name llm-api-app --resource-group rg-llm-demo --set-env-vars AZURE_OPENAI_KEY=secretref:oai-key MODEL_NAME=gpt-4o-mini

--- 5. Get public URL ---
az containerapp show --name llm-api-app --resource-gr

## Step 2: Auto-Scaling — How It Works

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Scaling rules: what triggers a scale-up?
scaling_rules = {
    "HTTP concurrency": {
        "trigger": "concurrent requests > 10 per replica",
        "command": "az containerapp update --scale-rule-type http --scale-rule-http-concurrency 10",
        "best_for": "Web APIs (our case)"
    },
    "CPU utilization": {
        "trigger": "CPU > 70% for 60s",
        "command": "az containerapp update --scale-rule-type azure-monitor (CPU metric)",
        "best_for": "Compute-heavy tasks"
    },
    "Queue depth": {
        "trigger": "Azure Service Bus queue length > 5",
        "command": "az containerapp update --scale-rule-type azure-servicebus",
        "best_for": "Batch LLM processing"
    }
}

print("Scaling Rule Options:")
for rule, info in scaling_rules.items():
    print(f"\n  {rule}:")
    for k,v in info.items():
        print(f"    {k}: {v}")

# Visualize scale-to-zero economics
np.random.seed(42)
t = np.arange(0, 120, 1)
requests = np.clip(40*np.sin(t*0.1)+20+np.random.normal(0,5,120), 0, 80)
replicas = np.clip(np.ceil(requests/10).astype(int), 0, 10)
# Cost: $0.000024 per vCPU-second, 0.5 vCPU per replica
cost_per_min = replicas * 0.5 * 0.000024 * 60

fig, axes = plt.subplots(3,1,figsize=(12,8), sharex=True)
fig.suptitle("Azure Container Apps: Scale-to-Zero Economics", fontsize=13, fontweight="bold")

axes[0].fill_between(t, requests, alpha=0.4, color="#0078D4")
axes[0].plot(t, requests, color="#0078D4", linewidth=2)
axes[0].set_ylabel("Concurrent Requests")
axes[0].axhline(10, color="orange", linestyle="--", label="Scale trigger (10 req/replica)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].step(t, replicas, color="#FF8C00", linewidth=2, where="post")
axes[1].fill_between(t, replicas, alpha=0.3, color="#FF8C00", step="post")
axes[1].set_ylabel("Active Replicas")
axes[1].grid(True, alpha=0.3)

axes[2].fill_between(t, cost_per_min*100, alpha=0.4, color="#107C10")
axes[2].plot(t, cost_per_min*100, color="#107C10", linewidth=2)
axes[2].set_ylabel("Cost (¢/min)")
axes[2].set_xlabel("Time (seconds)")
axes[2].grid(True, alpha=0.3)
idle_pct = (replicas==0).sum()/len(replicas)*100
axes[2].set_title(f"Idle {idle_pct:.0f}% of time = {idle_pct:.0f}% of time costs $0", fontsize=10)

plt.tight_layout()
plt.savefig("scaling_economics.png", dpi=100, bbox_inches="tight")
plt.close()
print("\n✅ Chart saved: scaling_economics.png")
print(f"   Simulation: {idle_pct:.0f}% idle time → zero cost during those periods")


Scaling Rule Options:

  HTTP concurrency:
    trigger: concurrent requests > 10 per replica
    best_for: Web APIs (our case)

  CPU utilization:
    trigger: CPU > 70% for 60s
    best_for: Compute-heavy tasks

  Queue depth:
    trigger: Azure Service Bus queue length > 5
    best_for: Batch LLM processing

✅ Chart saved: scaling_economics.png
   Simulation: 38% idle time → zero cost during those periods


## Step 3: Azure API Management — Products, Subscriptions, Routing

In [ ]:
# APIM concepts students need to understand:
# Product    = a bundle of APIs you offer (e.g. "Free Tier", "Pro Tier")
# Subscription = a client's access key to a Product
# Policy     = rules applied to every request (auth, rate limit, transform)

apim_concepts = """
                    ┌─────────────────────────────────────┐
                    │        Azure API Management          │
                    │                                      │
  Client Request    │  ┌──────────┐    ┌─────────────┐   │
  ─────────────────►│  │ Product  │    │   Policy    │   │
  + Ocp-Apim-       │  │ Free:    │    │ • Auth JWT  │   │
    Subscription-   │  │  100/min │    │ • Rate limit│   │
    Key: abc123     │  │ Pro:     │    │ • Log tokens│   │
                    │  │  1000/min│    │ • Route     │   │
                    │  └──────────┘    └──────┬──────┘   │
                    └─────────────────────────┼───────────┘
                                              │
                              ┌───────────────▼──────────────┐
                              │   Azure Container Apps        │
                              │   (llm-api-app)               │
                              └──────────────────────────────┘
"""
print(apim_concepts)

# APIM Policy XML — the core of routing and security
apim_policy = """
<policies>
  <inbound>
    <base />

    <!-- 1. Validate JWT — only authenticated clients pass -->
    <validate-jwt header-name="Authorization" failed-validation-httpcode="401"
                  failed-validation-error-message="Valid JWT required">
      <openid-config url="https://login.microsoftonline.com/{tenant}/.well-known/openid-configuration"/>
    </validate-jwt>

    <!-- 2. Rate limiting per subscription key -->
    <rate-limit-by-key calls="100" renewal-period="60"
                       counter-key="@(context.Subscription.Id)" />

    <!-- 3. Daily token quota (cost control) -->
    <quota-by-key calls="50000" renewal-period="86400"
                  counter-key="@(context.Subscription.Id)" />

    <!-- 4. Route to Container App backend -->
    <set-backend-service
      base-url="https://llm-api-app.proudbeach-12345.eastus.azurecontainerapps.io" />

    <!-- 5. Log token usage for billing (custom header forwarded to app) -->
    <set-header name="X-Request-Id" exists-action="override">
      <value>@(context.RequestId.ToString())</value>
    </set-header>
  </inbound>

  <outbound>
    <base />
    <set-header name="X-Powered-By" exists-action="override">
      <value>Azure APIM</value>
    </set-header>
  </outbound>

  <on-error>
    <base />
    <return-response>
      <set-status code="@(context.Response.StatusCode)" />
      <set-body>@(context.LastError.Message)</set-body>
    </return-response>
  </on-error>
</policies>
"""
print("APIM Policy XML:")
print(apim_policy)


[APIM architecture diagram printed]
APIM Policy XML: [rate limiting + JWT auth + routing printed]


In [ ]:
# APIM — Create Products and Subscriptions
# Run these commands in Azure Cloud Shell

RG       = 'rg-llm-demo'
APIM     = 'apim-llm-gateway'
APP_URL  = 'https://llm-api-app.proudbeach-12345.eastus.azurecontainerapps.io'

# Build commands as plain strings to avoid nested-quote issues
free_desc = '100 req/min, 50k tokens/day'
pro_desc  = '1000 req/min, unlimited tokens'

steps = [
    ('1. Create APIM instance (Developer tier — cheapest, ~$0.07/hr)',
     f'az apim create --name {APIM} --resource-group {RG}'
     ' --publisher-name LLM-Platform --publisher-email admin@example.com'
     ' --sku-name Developer --no-wait'),

    ('2. Create Free product (100 calls/min)',
     f'az apim product create --resource-group {RG} --service-name {APIM}'
     f' --product-id free-tier --product-name Free'
     f' --description "{free_desc}"'
     ' --state published --subscription-required true'),

    ('3. Create Pro product (1000 calls/min)',
     f'az apim product create --resource-group {RG} --service-name {APIM}'
     f' --product-id pro-tier --product-name Pro'
     f' --description "{pro_desc}"'
     ' --state published --subscription-required true'),

    ('4. Create API and link backend to Container App',
     f'az apim api create --resource-group {RG} --service-name {APIM}'
     f' --api-id llm-api --display-name LLM-API'
     f' --service-url {APP_URL} --path /api --protocols https'),

    ('5. Add API to Free product',
     f'az apim product api add --resource-group {RG} --service-name {APIM}'
     ' --product-id free-tier --api-id llm-api'),
]

print('='*70)
print('  APIM — Create Products and Subscriptions')
print('='*70)
for title, cmd in steps:
    print(f'\n--- {title} ---')
    print(cmd)

print()
print('[SYNTHETIC DEMO OUTPUT]')
print(f'APIM {APIM} provisioning (takes ~30 min)...')
print('Product "Free" created — 100 req/min, subscription required')
print('Product "Pro" created — 1000 req/min, subscription required')
print(f'API llm-api created, backend: {APP_URL}')
print('API added to Free product — subscribers can now call via APIM')


  APIM — Create Products and Subscriptions

--- 1. Create APIM instance (Developer tier — cheapest, ~$0.07/hr) ---
az apim create --name apim-llm-gateway --resource-group rg-llm-demo --publisher-name LLM-Platform --publisher-email admin@example.com --sku-name Developer --no-wait

--- 2. Create Free product (100 calls/min) ---
az apim product create --resource-group rg-llm-demo --service-name apim-llm-gateway --product-id free-tier --product-name Free --description '100 req/min, 50k tokens/day' --state published --subscription-required true

--- 3. Create Pro product (1000 calls/min) ---
az apim product create --resource-group rg-llm-demo --service-name apim-llm-gateway --product-id pro-tier --product-name Pro --description '1000 req/min, unlimited tokens' --state published --subscription-required true

--- 4. Create API and link backend ---
az apim api create --resource-group rg-llm-demo --service-name apim-llm-gateway --api-id llm-api --display-name LLM-API --service-url https://llm-a

## Step 4: End-to-End Test Through APIM

In [ ]:
import httpx, asyncio, json
import nest_asyncio
nest_asyncio.apply()

# Simulate an end-to-end API call through APIM
APIM_BASE = "https://apim-llm-gateway.azure-api.net"
SUBSCRIPTION_KEY = "your-subscription-key-from-apim-portal"

async def test_apim_call():
    print("End-to-End API Call: Client → APIM → Container App → LLM")
    print("="*60)

    # In a real test, use your actual APIM URL and subscription key
    # Here we show what the request/response looks like
    demo_request = {
        "url": f"{APIM_BASE}/api/v2/chat",
        "headers": {
            "Ocp-Apim-Subscription-Key": "abc123...  (from APIM portal)",
            "Authorization": "Bearer eyJ...  (JWT token)",
            "Content-Type": "application/json"
        },
        "body": {
            "messages": [{"role": "user", "content": "What is Azure ML?"}],
            "temperature": 0.7
        }
    }
    demo_response = {
        "status": 200,
        "headers": {"X-Powered-By": "Azure APIM", "X-Request-Id": "req-a1b2c3d4"},
        "body": {
            "id": "chatcmpl-demo",
            "content": "Azure Machine Learning is a cloud-based platform...",
            "model": "gpt-4o-mini",
            "usage": {"prompt_tokens": 15, "completion_tokens": 45, "total_tokens": 60},
            "latency_ms": 843.2
        }
    }
    demo_rate_limit = {
        "status": 429,
        "error": "Rate limit exceeded. Retry after 60 seconds.",
        "headers": {"Retry-After": "60"}
    }

    print(f"\n📤 REQUEST:")
    print(f"  POST {demo_request['url']}")
    for k,v in demo_request["headers"].items():
        print(f"  {k}: {v}")
    print(f"  Body: {json.dumps(demo_request['body'])}")

    print(f"\n📥 RESPONSE (200 OK):")
    print(json.dumps(demo_response, indent=2))

    print(f"\n📥 RESPONSE (429 — rate limited after 101st call):")
    print(json.dumps(demo_rate_limit, indent=2))

asyncio.run(test_apim_call())


End-to-End API Call: Client → APIM → Container App → LLM

📤 REQUEST:
  POST https://apim-llm-gateway.azure-api.net/api/v2/chat
  Ocp-Apim-Subscription-Key: abc123...
  Authorization: Bearer eyJ...

📥 RESPONSE (200 OK):
  {"id":"chatcmpl-demo","content":"Azure Machine Learning is...","latency_ms":843.2}

📥 RESPONSE (429 — rate limited after 101st call):
  {"status":429,"error":"Rate limit exceeded. Retry after 60 seconds."}


## Step 5: Canary Deployment — Traffic Splitting

In [ ]:
# Canary Deployment
# Run commands in Azure Cloud Shell.

RG = 'rg-llm-demo'
APP = 'llm-api-app'
ACR = 'acrllmdemo'

steps = [
    ('1. Deploy v2 as new revision', f'az containerapp revision copy --name {APP} --resource-group {RG} --image {ACR}.azurecr.io/llm-api:v2.0 --revision-suffix v2'),
    ('2. Split: 90% stable v1, 10% canary v2', f'az containerapp ingress traffic set --name {APP} --resource-group {RG} --revision-weight {APP}--v1=90 {APP}--v2=10'),
    ('3. Monitor v2 error rate for 30 min, then promote', f'az containerapp ingress traffic set --name {APP} --resource-group {RG} --revision-weight {APP}--v2=100'),
    ('4. Rollback if v2 has errors', f'az containerapp ingress traffic set --name {APP} --resource-group {RG} --revision-weight {APP}--v1=100'),
]
print("="*70)
print("  Canary Deployment")
print("="*70)
for t,c in steps:
    print(f"\n--- {t} ---")
    print(c)
print('Revision llm-api-app--v2 created')
print('Traffic split: v1=90%, v2=10%')
print('Monitoring for 30 min: v2 error_rate=0.2%, P99=810ms ✅')
print('Promoted: v2=100% — deployment complete')
print('📌 Rollback command ready if needed')


  Canary Deployment

--- 1. Deploy v2 as new revision ---
az containerapp revision copy --name llm-api-app --resource-group rg-llm-demo --image acrllmdemo.azurecr.io/llm-api:v2.0 --revision-suffix v2

--- 2. Split: 90% stable v1, 10% canary v2 ---
az containerapp ingress traffic set --name llm-api-app --resource-group rg-llm-demo --revision-weight llm-api-app--v1=90 llm-api-app--v2=10

--- 3. Monitor v2 error rate for 30 min, then promote ---
az containerapp ingress traffic set --name llm-api-app --resource-group rg-llm-demo --revision-weight llm-api-app--v2=100

--- 4. Rollback if v2 has errors ---
az containerapp ingress traffic set --name llm-api-app --resource-group rg-llm-demo --revision-weight llm-api-app--v1=100

[SYNTHETIC DEMO OUTPUT]
Revision llm-api-app--v2 created
Traffic split: v1=90%, v2=10%
Monitoring for 30 min: v2 error_rate=0.2%, P99=810ms ✅
Promoted: v2=100% — deployment complete
📌 Rollback command ready if needed


In [ ]:
print("📌 Key Takeaways:")
print("  • Container Apps: min-replicas=0 = free when idle (use for dev/staging)")
print("  • Always set secrets via Key Vault ref — NEVER as plain env vars in prod")
print("  • APIM = single gateway: auth, rate limits, routing, observability")
print("  • APIM Products let you offer different tiers (Free/Pro) to different clients")
print("  • Canary deployments: ship to 10% → validate → promote to 100%")
print("  • Rollback = 30 seconds with az containerapp ingress traffic set")


📌 Key Takeaways:
  • Container Apps: min-replicas=0 = free when idle (use for dev/staging)
  • Always set secrets via Key Vault ref — NEVER as plain env vars in prod
  • APIM = single gateway: auth, rate limits, routing, observability
  • APIM Products let you offer different tiers (Free/Pro) to different clients
  • Canary deployments: ship to 10% → validate → promote to 100%
  • Rollback = 30 seconds with az containerapp ingress traffic set
